# Quickstart: score a first submission

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jean-jsj/CARD/blob/main/examples/01_quickstart.ipynb)

Build the simplest possible submission and score it against the hidden truth. Runs in a few minutes on the ~18 MB starter slice; on Colab this notebook is self-contained.

In [1]:
import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules and not Path("card_metrics").is_dir():
    subprocess.run(["git", "clone", "-q", "https://github.com/jean-jsj/CARD"], check=True)
    os.chdir("CARD")
elif Path.cwd().name == "examples":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd

Download the starter slice of the endogeneity-on cell (10 of 731 stores, all 40 products).

In [2]:
from huggingface_hub import snapshot_download

CELL = "complex_log_log_endogenous_seed001"
snapshot_download(repo_id="jean-jsj/CARD", repo_type="dataset",
                  allow_patterns=[f"dev_mini/{CELL}/*"], local_dir="benchmark")
cell_dir = Path("benchmark/dev_mini") / CELL

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

A cell has `public/` (everything your model may use) and `hidden/` (scoring truth, dev seed only — never model input). The training panel is one row per product, store, and week with positive sales:

In [3]:
train = pd.read_csv(cell_dir / "public/transactions_train_public.csv")
train.head()

,product_id,store_id,week,units,dollars,price,promo_flag,promo_cost,supply_cost_proxy
0,P001,S0001,1427,287.0,513.73,1.79,0,0.0,1.0069
1,P002,S0001,1427,26.0,28.34,1.09,0,0.0,0.6072
2,P003,S0001,1427,91.0,181.09,1.99,0,0.0,1.0279
3,P004,S0001,1427,59.0,99.71,1.69,0,0.0,1.0241
4,P005,S0001,1427,10.0,23.90,2.39,0,0.0,1.2248


`promo_cost` and `supply_cost_proxy` are the two released price instruments. Each product also ships marketing copy — the carrier of the substitution structure:

In [4]:
products = pd.read_csv(cell_dir / "public/products_public.csv")
products.sample(2, random_state=0)

,product_id,product_text,brand_code
22,P023,"The surface is soft and smooth to the hand, ge...",B2
20,P021,A soft and gentle touch meets the skin with a ...,B8


A submission is up to three CSVs ([format](../docs/SUBMISSION_FORMAT.md)). This naive floor predicts recent average sales, zero elasticities, and zero demand response to any price change.

In [5]:
from itertools import product as iproduct

holdout = pd.read_csv(cell_dir / "public/transactions_holdout_context_public.csv")
sweep = pd.read_csv(cell_dir / "public/counterfactual_sweep_context_public.csv")
sub = Path("submissions_local/naive") / CELL
sub.mkdir(parents=True, exist_ok=True)

recent = train[train.week > train.week.max() - 8]
mean_units = recent.groupby(["product_id", "store_id"])["units"].mean().rename("predicted_units")
holdout[["product_id", "store_id", "week"]].merge(mean_units, on=["product_id", "store_id"],
    how="left").fillna(0.0).to_csv(sub / "forecast_predictions.csv", index=False)

ids = products.product_id.tolist()
pd.DataFrame([(j, i, 0.0) for j, i in iproduct(ids, ids)],
    columns=["priced_product_id", "affected_product_id", "elasticity"]
    ).to_csv(sub / "elasticity_matrix.csv", index=False)

sweep[["intervention_id", "product_id", "store_id", "week"]].drop_duplicates().assign(
    predicted_delta_units=0.0).to_csv(sub / "counterfactual_deltas.csv", index=False)

Score it. On the dev seed this is local and instant.

In [6]:
import json

out = Path("submissions_local/naive_score.json")
subprocess.run([sys.executable, "-m", "card_metrics.evaluate_submission",
                "--cell-dir", str(cell_dir), "--submission-dir", str(sub),
                "--submission-name", "naive", "--out", str(out)], check=True)
score = json.loads(out.read_text())

{"own-price bias (ranked, 0 = unbiased)": score["counterfactual_prediction"]["headline"]["own_price"]["own_price_wmpe"],
 "substitution error (1 = no change predicted)": score["counterfactual_prediction"]["headline"]["substitution"]["substitution_wape"],
 "forecast error (WMAPE)": round(score["sales_forecasting"]["demand_wmape"], 3)}

wrote submissions_local/naive_score.json


{'own-price bias (ranked, 0 = unbiased)': 1.0,
 'substitution error (1 = no change predicted)': 1.0,
 'forecast error (WMAPE)': 0.544}

Predicting "prices don't matter" scores own-price bias 1.0: the error is the entire true demand response. The leaderboard ranks |own-price bias|; the forecast error is displayed beside it but never ranked — the benchmark's point is that the two diverge.

Next: [02_endogeneity.ipynb](02_endogeneity.ipynb) shows that divergence live. To enter the leaderboard, score the full cells and open a PR ([CONTRIBUTING](../.github/CONTRIBUTING.md)).